# ByteEmbed — STS-focused study (byte students)

Tests whether **byte-level** embeddings work for STS *when trained for it* (our main distillation runs use a retrieval-leaning objective that under-serves STS). Supervised STS fine-tuning with the **CoSENT** rank objective — the modern STS standard, which directly optimizes the rank correlation Spearman measures.

**8 languages, all with real native graded `sentence1/sentence2/score` train data:**
- **6 low-resource:** Amharic, Hausa, Kinyarwanda, Telugu, Marathi, Algerian Arabic (SemRel24STS)
- **2 anchors:** English (SemRel24STS), Chinese (C-MTEB/STSB)

Train on `train`+`dev`, **evaluate on `test` only (leakage-safe)**. Each byte student is **warm-started from its SONAR-distilled checkpoint** (strong multilingual base) then STS-specialized. Reports per-language Spearman, **raw and with whitening** (the standard post-hoc STS boost). Run top-to-bottom; do the smoke cell first.

### 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

### 2. Clone repo + install deps

In [ ]:
import os
os.chdir('/content')
REPO = 'https://github.com/Aarushvinod/embedding-research.git'
if not os.path.isdir('/content/embedding-research'):
    !git clone -q $REPO
os.chdir('/content/embedding-research')
!git pull -q
!pip install -q -r requirements-cloud.txt
!pip install -q -e . || echo '(editable install skipped — running from repo root is fine)'
print('setup done | cwd', os.getcwd())

### 3. Persist to Drive
Point at the **same** `byteembed_lowres` folder as the distillation runs — so the byte students warm-start from their SONAR-distilled checkpoints (`checkpoints/byte-*_attn.pt`). Skip this cell to run on ephemeral disk (students then train from the raw byt5 base).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
PERSIST = '/content/drive/MyDrive/byteembed_lowres'   # SAME folder as the distillation runs
for d in ('results', 'checkpoints'):
    os.makedirs(f'{PERSIST}/{d}', exist_ok=True)
    if not os.path.islink(d):
        if os.path.isdir(d): shutil.rmtree(d)
        os.symlink(f'{PERSIST}/{d}', d)
print('persisting results/ and checkpoints/ to', PERSIST)

### 4. Smoke test (~5 min) — validate the whole STS pipeline
3 languages (am/rw/en), one tiny byte student, 80 steps. If it prints a Spearman table, everything works (STS data → CoSENT train → test-only eval → whitening).

In [ ]:
from byte_embed.run_sts import run
_ = run(family='byte', smoke=True, out='results/sts_byte_smoke.json')

### 5. Full STS run — byte small / base / large
**CoSENT** fine-tuning, **mean pooling** (classic for STS + whitening), 2000 steps. Warm-starts each size from its distilled checkpoint when present (else trains from the raw byt5 base). Resumable; eval is test-only; whitening reported alongside raw.

In [ ]:
from byte_embed.run_sts import run
_ = run(
    family='byte',
    sizes=('small', 'base', 'large'),   # large warm-starts only if checkpoints/byte-large_attn.pt exists
    pooling='mean',
    steps=2000,
    whiten=True,
    out='results/sts_byte.json',
)

### 6. Results — per-language Spearman (raw + whitened)

In [ ]:
import json
from byte_embed.run_sts import _summary
_summary(json.load(open('results/sts_byte.json')))

### 7. Download results

In [ ]:
from google.colab import files
files.download('results/sts_byte.json')